In [1]:
#1. Import the libraries, classes, modules needed for the project

import pandas as pd
import numpy as np
from copy import deepcopy
from collections import defaultdict
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import os
import json
import pickle
from itertools import combinations


In [2]:
# 2. Read the data files into dataframes from the extracted raw data folder
df_matches = pd.read_csv("../data/raw/matches.csv")
df_teams = pd.read_csv("../data/raw/teams.csv")
df_tournaments = pd.read_csv("../data/raw/tournaments.csv")

print(f"Matches Data is loaded with entries: {df_matches.shape}")
print(f"Teams Data is loaded with entries: {df_teams.shape}")
print(f"Tournaments Data is loaded with entries: {df_tournaments.shape}")

Matches Data is loaded with entries: (1068, 28)
Teams Data is loaded with entries: (537, 16)
Tournaments Data is loaded with entries: (23, 4)


In [38]:
# 2. Create the Class to have the whole project start here
class WorldCupPredictor:

    def __init__(self):
        self.df_matches = None
        self.df_teams = None
        self.df_tournaments = None
        self.training_dataset = None
        self.model = None
        self.feature_columns = None
        self.current_elo = {}
        self.team_history = {}
        self.h2h_history = {}
        self.X = None
        self.y = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.train_years = None
        self.test_years = None
        self.training_history = None
        self.y_pred = None
        self.y_pred_proba = None
        self.evaluation_results = None
        self.feature_importance_df = None
        self.top_features = None
        self.team_snapshots = {}
        self.team_snapshots = {}
        self.last_prediction = None
        self.teams_2026 = None
        self.groups = None
        self.group_tables = None
        self.knockout_bracket = None
        self.stage_weights = {
            "group": 1,
            "round_of_32": 2,
            "round_of_16": 3,
            "quarter_final": 4,
            "semi_final": 5,
            "third_place": 5,
            "final": 6
        }
        self.groups = {}

        self.group_matches = {}

        self.group_tables = {}

        self.group_match_results = {}

        self.third_place_table = None

        self.round_of_32 = []

        self.round_of_16 = []

        self.quarter_finals = []

        self.semi_finals = []

        self.final = None

        self.third_place_match = None

        self.fifa_rankings = {}

    # 1. Load the data and confirm
    def load_data(self, df_matches, df_teams, df_tournaments):
        self.df_matches = df_matches.copy()
        self.df_teams = df_teams.copy()
        self.df_tournaments = df_tournaments.copy()

        print("Data loaded successfully")
        print(f"Matches: {self.df_matches.shape}")
        print(f"Teams: {self.df_teams.shape}")
        print(f"Tournaments: {self.df_tournaments.shape}")

    # 4. Clean the data, remove spaces, set the right date formats, set all integers to floats
    def clean_data(self):
        # Dates
        self.df_matches["date"] = pd.to_datetime(
            self.df_matches["date"],
            errors="coerce"
        )

        # Scores
        self.df_matches["homeScore"] = pd.to_numeric(
            self.df_matches["homeScore"],
            errors="coerce"
        )
        self.df_matches["awayScore"] = pd.to_numeric(
            self.df_matches["awayScore"],
            errors="coerce"
        )

        # Strip spaces
        string_columns = ["homeTeam", "awayTeam", "stage", "country"]
        for col in string_columns:
            self.df_matches[col] = (
                self.df_matches[col]
                .astype(str)
                .str.strip()
            )

        self.df_teams["name"] = (
            self.df_teams["name"]
            .astype(str)
            .str.strip()
        )
        self.df_teams["confederation"] = (
            self.df_teams["confederation"]
            .astype(str)
            .str.strip()
        )

        # Fill missing scores
        self.df_matches["homeScore"] = self.df_matches["homeScore"].fillna(0)
        self.df_matches["awayScore"] = self.df_matches["awayScore"].fillna(0)

        print("Cleaning completed")

    # 5. Remove the matches from 2026 because they are what we are predicting
    def remove_future_matches(self, prediction_year=2026):
        before = len(self.df_matches)
        self.df_matches = self.df_matches[
            self.df_matches["tournament_year"] < prediction_year
        ].copy()
        after = len(self.df_matches)

        print(f"Removed {before-after} future matches")
        print(f"Remaining matches: {after}")

    # 6. Rename some fields due to changes in country names in the world cup history
    def standardize_team_names(self):
        mapping = {
            "West Germany": "Germany",
            "FR Yugoslavia": "Serbia",
            "Serbia and Montenegro": "Serbia",
            "Soviet Union": "Russia",
            "Czechoslovakia": "Czech Republic"
        }

        self.df_matches["homeTeam"] = self.df_matches["homeTeam"].replace(mapping)
        self.df_matches["awayTeam"] = self.df_matches["awayTeam"].replace(mapping)
        self.df_teams["name"] = self.df_teams["name"].replace(mapping)

        print("Team names standardized")

    # Check the data that its loaded, cleaned and formatted to the right types, renamed fields to capture change of country names
    def validate_data(self):
        print("="*50)
        print("MATCHES")
        print(self.df_matches.shape)
        print(self.df_matches.isnull().sum())

        print("="*50)
        print("TEAMS")
        print(self.df_teams.shape)
        print(self.df_teams.isnull().sum())

        print("="*50)
        print("TOURNAMENTS")
        print(self.df_tournaments.shape)
        print(self.df_tournaments.isnull().sum())

    def normalize_stages(self):
        # Creates ML-friendly stage features while preserving the original tournament structure
        stage_map = {
            # Group Stage
            "group_1": "group",
            "group_2": "group",
            "group_3": "group",
            "group_4": "group",
            "group_5": "group",
            "group_6": "group",
            "group_7": "group",
            "group_8": "group",

            # Knockout
            "r32": "round_of_32",
            "r16": "round_of_16",
            "qf": "quarter_final",
            "sf": "semi_final",
            "final": "final",
            "third_place": "third_place"
        }

        # Preserve original stage
        self.df_matches["stage_category"] = (
            self.df_matches["stage"]
            .astype(str)
            .str.lower()
            .map(stage_map)
        )

        # Extract group number
        self.df_matches["group_number"] = (
            self.df_matches["stage"]
            .astype(str)
            .str.extract(r'group_(\d+)')[0]
        )

        print("Stage normalization completed.")
        print(self.df_matches[["stage", "stage_category", "group_number"]].head(20))

    # Build  target and separate home and away
    def build_target(self):
        """
        Creates target variable.
        Home Win = 1
        Draw = 0
        Away Win = -1
        """

        self.df_matches["target"] = np.where(
            self.df_matches["homeScore"] > self.df_matches["awayScore"],
            1,
            np.where(
                self.df_matches["homeScore"] < self.df_matches["awayScore"],
                -1,
                0
            )
        )

        print("Target variable created.")
        print(
            self.df_matches[
                [
                    "homeTeam",
                    "awayTeam",
                    "homeScore",
                    "awayScore",
                    "target"
                ]
            ].head(10)
        )

    #sort the matches in ascending order to help calculate for rolling features
    def sort_matches(self):

        self.df_matches = (
            self.df_matches
            .sort_values(by=["date"])
            .reset_index(drop=True)
        )

        print("Matches sorted chronologically.")
        print(
            self.df_matches[
                [
                    "date",
                    "homeTeam",
                    "awayTeam"
                ]
            ].head()
        )

    # Calculate the Elo Ratings
    def calculate_elo(self, k_factor=30, initial_elo=1500):
        """
        Calculate pre-match Elo ratings.

        World Cup matches are treated as neutral venue games.

        Features created:
        -----------------
        home_team_elo
        away_team_elo
        elo_difference
        """

        elo = {}

        home_elos = []
        away_elos = []
        elo_differences = []

        for _, match in self.df_matches.iterrows():
            team1 = match["homeTeam"]
            team2 = match["awayTeam"]

            # Initialize ratings
            if team1 not in elo:
                elo[team1] = initial_elo

            if team2 not in elo:
                elo[team2] = initial_elo

            # Store ratings BEFORE match
            team1_elo = elo[team1]
            team2_elo = elo[team2]

            home_elos.append(team1_elo)
            away_elos.append(team2_elo)
            elo_differences.append(team1_elo - team2_elo)

            # Expected result
            expected_team1 = 1 / (
                1 + 10 ** ((team2_elo - team1_elo) / 400)
            )
            expected_team2 = 1 - expected_team1

            # Actual result
            if match["homeScore"] > match["awayScore"]:
                actual_team1 = 1
                actual_team2 = 0
            elif match["homeScore"] < match["awayScore"]:
                actual_team1 = 0
                actual_team2 = 1
            else:
                actual_team1 = 0.5
                actual_team2 = 0.5

            # Update Elo
            elo[team1] = (
                team1_elo
                + k_factor * (actual_team1 - expected_team1)
            )
            elo[team2] = (
                team2_elo
                + k_factor * (actual_team2 - expected_team2)
            )

        # Save features (unindented from the for loop, aligned with the method body)
        self.df_matches["home_team_elo"] = home_elos
        self.df_matches["away_team_elo"] = away_elos
        self.df_matches["elo_difference"] = elo_differences

        # Save latest ratings for tournament simulation
        self.current_elo = elo

        print("Elo ratings calculated.")
        print(
            self.df_matches[
                [
                    "homeTeam",
                    "awayTeam",
                    "home_team_elo",
                    "away_team_elo",
                    "elo_difference"
                ]
            ].head(10)
        )
        self.current_elo = elo

    # helper function retrieving the current Elo rating
    def get_current_elo(self, team):

        return self.current_elo.get(team, 1500)

    # the recent 5 match results 'recent matches team form'
    def build_rolling_features(self, window=5):
        """
        Build rolling World Cup form features.

        Uses only matches BEFORE the current match.
        Ignores API home/away designation.
        """

        from collections import defaultdict

        team_history = defaultdict(list)

        home_win_rate = []
        away_win_rate = []
        home_goals = []
        away_goals = []
        home_conceded = []
        away_conceded = []
        home_goal_diff = []
        away_goal_diff = []

        for _, match in self.df_matches.iterrows():
            team1 = match["homeTeam"]
            team2 = match["awayTeam"]

            # ---------------------
            # TEAM 1 FEATURES
            # ---------------------
            recent = team_history[team1][-window:]

            if len(recent) == 0:
                home_win_rate.append(0)
                home_goals.append(0)
                home_conceded.append(0)
                home_goal_diff.append(0)
            else:
                home_win_rate.append(
                    np.mean([x["result"] for x in recent])
                )
                home_goals.append(
                    np.mean([x["gf"] for x in recent])
                )
                home_conceded.append(
                    np.mean([x["ga"] for x in recent])
                )
                home_goal_diff.append(
                    np.mean([x["gf"] - x["ga"] for x in recent])
                )

            # ---------------------
            # TEAM 2 FEATURES
            # ---------------------
            recent = team_history[team2][-window:]

            if len(recent) == 0:
                away_win_rate.append(0)
                away_goals.append(0)
                away_conceded.append(0)
                away_goal_diff.append(0)
            else:
                away_win_rate.append(
                    np.mean([x["result"] for x in recent])
                )
                away_goals.append(
                    np.mean([x["gf"] for x in recent])
                )
                away_conceded.append(
                    np.mean([x["ga"] for x in recent])
                )
                away_goal_diff.append(
                    np.mean([x["gf"] - x["ga"] for x in recent])
                )

            # ---------------------
            # UPDATE TEAM HISTORY
            # ---------------------
            hs = match["homeScore"]
            aw = match["awayScore"]

            if hs > aw:
                team1_result = 1
                team2_result = 0
            elif hs < aw:
                team1_result = 0
                team2_result = 1
            else:
                team1_result = 0.5
                team2_result = 0.5

            team_history[team1].append({
                "result": team1_result,
                "gf": hs,
                "ga": aw
            })

            team_history[team2].append({
                "result": team2_result,
                "gf": aw,
                "ga": hs
            })

        # Save features (Outside the for-loop)
        self.df_matches["home_win_rate_last_5"] = home_win_rate
        self.df_matches["away_win_rate_last_5"] = away_win_rate
        self.df_matches["home_goals_last_5"] = home_goals
        self.df_matches["away_goals_last_5"] = away_goals
        self.df_matches["home_conceded_last_5"] = home_conceded
        self.df_matches["away_conceded_last_5"] = away_conceded
        self.df_matches["home_goal_diff_last_5"] = home_goal_diff
        self.df_matches["away_goal_diff_last_5"] = away_goal_diff

        # Save latest history mapping for simulation states
        self.team_history = team_history

        print("Rolling features calculated.")
        print(
            self.df_matches[
                [
                    "homeTeam",
                    "awayTeam",
                    "home_win_rate_last_5",
                    "away_win_rate_last_5",
                    "home_goals_last_5",
                    "away_goals_last_5"
                ]
            ].head(10)
        )

    # build a head to head team pairings
    def build_head_to_head(self):
        h2h_history = defaultdict(list)

        h2h_games = []
        h2h_win_rate = []
        h2h_goal_diff = []

        for _, match in self.df_matches.iterrows():
            team1 = match["homeTeam"]
            team2 = match["awayTeam"]

            pair = tuple(sorted([team1, team2]))
            previous = h2h_history[pair]

            # -----------------------
            # BUILD FEATURES
            # -----------------------
            if len(previous) == 0:
                h2h_games.append(0)
                h2h_win_rate.append(0.5)
                h2h_goal_diff.append(0)
            else:
                h2h_games.append(len(previous))
                h2h_win_rate.append(
                    np.mean([x["result"] for x in previous])
                )
                h2h_goal_diff.append(
                    np.sum([x["goal_diff"] for x in previous])
                )

            # -----------------------
            # UPDATE HISTORY
            # -----------------------
            hs = match["homeScore"]
            aw = match["awayScore"]

            if pair[0] == team1:
                goal_diff = hs - aw
                if hs > aw:
                    result = 1
                elif hs < aw:
                    result = 0
                else:
                    result = 0.5
            else:
                goal_diff = aw - hs
                if aw > hs:
                    result = 1
                elif aw < hs:
                    result = 0
                else:
                    result = 0.5

            h2h_history[pair].append({
                "result": result,
                "goal_diff": goal_diff
            })

        # Save features (Outside the for-loop)
        self.df_matches["head_to_head_games"] = h2h_games
        self.df_matches["head_to_head_win_rate"] = h2h_win_rate
        self.df_matches["head_to_head_goal_difference"] = h2h_goal_diff

        # Save historical lookup dictionary for out-of-sample simulation metrics
        self.h2h_history = h2h_history

        print("Head-to-head features calculated.")
        print(
            self.df_matches[
                [
                    "homeTeam",
                    "awayTeam",
                    "head_to_head_games",
                    "head_to_head_win_rate",
                    "head_to_head_goal_difference"
                ]
            ].head(10)
        )
    # Build the historical performance rating of teams
    def build_historical_features(self):

        home_titles = []
        away_titles = []
        home_knockout = []
        away_knockout = []
        home_avg_finish = []
        away_avg_finish = []
        home_apps = []
        away_apps = []
        home_sf = []
        away_sf = []
        home_final = []
        away_final = []

        for _, match in self.df_matches.iterrows():
            year = match["tournament_year"]
            team1 = match["homeTeam"]
            team2 = match["awayTeam"]

            # ------------------------
            # TEAM 1
            # ------------------------
            history = self.df_teams[
                (self.df_teams["name"] == team1)
                & (self.df_teams["tournament_year"] < year)
            ]

            apps = len(history)
            home_apps.append(apps)

            if apps == 0:
                home_titles.append(0)
                home_knockout.append(0)
                home_avg_finish.append(32)
                home_sf.append(0)
                home_final.append(0)
            else:
                home_titles.append(
                    (history["finalPosition"] == 1).sum()
                )
                home_avg_finish.append(
                    history["finalPosition"]
                    .fillna(32)
                    .mean()
                )

                knockout_count = (
                    history["knockoutPath"]
                    .fillna("[]")
                    .astype(str) != "[]"
                ).sum()
                home_knockout.append(knockout_count / apps)

                sf_count = (
                    history["knockoutPath"]
                    .fillna("")
                    .astype(str)
                    .str.contains("'stage': 'sf'")
                ).sum()
                home_sf.append(sf_count / apps)

                final_count = (
                    history["knockoutPath"]
                    .fillna("")
                    .astype(str)
                    .str.contains("'stage': 'final'")
                ).sum()
                home_final.append(final_count / apps)

            # ------------------------
            # TEAM 2
            # ------------------------
            history = self.df_teams[
                (self.df_teams["name"] == team2)
                & (self.df_teams["tournament_year"] < year)
            ]

            apps = len(history)
            away_apps.append(apps)

            if apps == 0:
                away_titles.append(0)
                away_knockout.append(0)
                away_avg_finish.append(32)
                away_sf.append(0)
                away_final.append(0)
            else:
                away_titles.append(
                    (history["finalPosition"] == 1).sum()
                )
                away_avg_finish.append(
                    history["finalPosition"]
                    .fillna(32)
                    .mean()
                )

                knockout_count = (
                    history["knockoutPath"]
                    .fillna("[]")
                    .astype(str) != "[]"
                ).sum()
                away_knockout.append(knockout_count / apps)

                sf_count = (
                    history["knockoutPath"]
                    .fillna("")
                    .astype(str)
                    .str.contains("'stage': 'sf'")
                ).sum()
                away_sf.append(sf_count / apps)

                final_count = (
                    history["knockoutPath"]
                    .fillna("")
                    .astype(str)
                    .str.contains("'stage': 'final'")
                ).sum()
                away_final.append(final_count / apps)

        # Save features back to DataFrame (Outside the loop)
        self.df_matches["home_historical_titles"] = home_titles
        self.df_matches["away_historical_titles"] = away_titles
        self.df_matches["home_knockout_rate"] = home_knockout
        self.df_matches["away_knockout_rate"] = away_knockout
        self.df_matches["home_average_finish"] = home_avg_finish
        self.df_matches["away_average_finish"] = away_avg_finish
        self.df_matches["home_world_cup_appearances"] = home_apps
        self.df_matches["away_world_cup_appearances"] = away_apps
        self.df_matches["home_semi_final_rate"] = home_sf
        self.df_matches["away_semi_final_rate"] = away_sf
        self.df_matches["home_final_rate"] = home_final
        self.df_matches["away_final_rate"] = away_final

        print("Historical features calculated.")
        print(
            self.df_matches[
                [
                    "homeTeam",
                    "awayTeam",
                    "home_historical_titles",
                    "away_historical_titles",
                    "home_world_cup_appearances",
                    "away_world_cup_appearances",
                    "home_final_rate",
                    "away_final_rate"
                ]
            ].head(10)
        )

    # adding weight to the tournament stages
    def build_stage_weight(self):

        stage_map = {
            "group_a": 1,
            "group_b": 1,
            "group_c": 1,
            "group_d": 1,
            "group_e": 1,
            "group_f": 1,
            "group_g": 1,
            "group_h": 1,
            "round_of_16": 2,
            "quarter_final": 3,
            "semi_final": 4,
            "third_place": 4,
            "final": 5
        }

        self.df_matches["stage_weight"] = (
            self.df_matches["stageNormalized"]
            .map(stage_map)
            .fillna(1)
        )

        print("Stage weights created.")
        print(
            self.df_matches[
                [
                    "stageNormalized",
                    "stage_weight"
                ]
            ]
            .drop_duplicates()
            .sort_values("stage_weight")
        )

    # Build a training data set
    def build_training_dataset(self):
        self.feature_columns = [
            "home_team_elo",
            "away_team_elo",
            "elo_difference",

            "home_win_rate_last_5",
            "away_win_rate_last_5",

            "home_goals_last_5",
            "away_goals_last_5",

            "home_conceded_last_5",
            "away_conceded_last_5",

            "home_goal_diff_last_5",
            "away_goal_diff_last_5",

            "head_to_head_games",
            "head_to_head_win_rate",
            "head_to_head_goal_difference",

            "home_historical_titles",
            "away_historical_titles",

            "home_knockout_rate",
            "away_knockout_rate",

            "home_average_finish",
            "away_average_finish",

            "home_world_cup_appearances",
            "away_world_cup_appearances",

            "home_semi_final_rate",
            "away_semi_final_rate",

            "home_final_rate",
            "away_final_rate",

            "stage_weight"
        ]

        required_columns = (
            self.feature_columns
            + [
                "target",
                "homeTeam",
                "awayTeam",
                "date",
                "tournament_year"
            ]
        )

        self.training_dataset = (
            self.df_matches[required_columns]
            .copy()
        )

        # Fill any missing feature values
        self.training_dataset[self.feature_columns] = (
            self.training_dataset[self.feature_columns]
            .fillna(0)
        )

        # Remove rows without target
        self.training_dataset = (
            self.training_dataset[
                self.training_dataset["target"].notna()
            ]
            .reset_index(drop=True)
        )

        print("=" * 60)
        print("Training dataset created")
        print("=" * 60)

        print(f"Rows: {self.training_dataset.shape[0]}")
        print(f"Columns: {self.training_dataset.shape[1]}")
        print()

        print("Feature columns:")
        for feature in self.feature_columns:
            print(feature)

        print()
        print(self.training_dataset.head())

    #Select features and separate them from the target
    def select_features(self):
        """
        Build feature matrix X and target vector y.
        """

        self.X = self.training_dataset[self.feature_columns].copy()
        self.y = self.training_dataset["target"].copy()

        print("=" * 60)
        print("Features selected")
        print("=" * 60)

        print(f"X Shape : {self.X.shape}")
        print(f"y Shape : {self.y.shape}")
        print()

        print("Feature Columns")
        for col in self.feature_columns:
            print(f"- {col}")

        print()
        print("Target Distribution")
        print(
            self.y.value_counts()
            .sort_index()
        )
        self.n_features = len(self.feature_columns)

        return self.X, self.y

    # Split the training and testing data training 1960 - 2014 training ---2018-2022 testing
    def split_data(self, test_tournaments=2):
        """
        Time-based split.

        Default:
        ----------
        Train : All but latest 2 tournaments
        Test  : Latest 2 tournaments
        """

        years = sorted(
            self.training_dataset["tournament_year"].unique()
        )

        self.test_years = years[-test_tournaments:]
        self.train_years = years[:-test_tournaments]

        train_mask = (
            self.training_dataset["tournament_year"].isin(
                self.train_years
            )
        )

        test_mask = (
            self.training_dataset["tournament_year"].isin(
                self.test_years
            )
        )

        self.X_train = self.training_dataset.loc[
            train_mask,
            self.feature_columns
        ]

        self.X_test = self.training_dataset.loc[
            test_mask,
            self.feature_columns
        ]

        self.y_train = self.training_dataset.loc[
            train_mask,
            "target"
        ]

        self.y_test = self.training_dataset.loc[
            test_mask,
            "target"
        ]

        print("=" * 60)
        print("Time-based split completed")
        print("=" * 60)
        print()

        print(f"Training tournaments : {self.train_years}")
        print(f"Testing tournaments  : {self.test_years}")
        print()

        print(f"X_train : {self.X_train.shape}")
        print(f"X_test  : {self.X_test.shape}")
        print()

        print(f"y_train : {self.y_train.shape}")
        print(f"y_test  : {self.y_test.shape}")

    #train the model using X_train, y_train
    def train_model(
        self,
        iterations=500,
        learning_rate=0.05,
        depth=6,
        early_stopping_rounds=50,
        random_seed=42
    ):
        """
        Train the World Cup prediction model.

        Stores:
            self.model
            self.class_labels
        """

        # -----------------------------
        # Validation
        # -----------------------------
        if self.X_train is None or self.y_train is None:
            raise ValueError(
                "Training data not found. Run split_data() first."
            )

        if self.X_test is None or self.y_test is None:
            raise ValueError(
                "Test data not found. Run split_data() first."
            )

        # -----------------------------
        # Initialize model
        # -----------------------------
        model = CatBoostClassifier(
            iterations=iterations,
            learning_rate=learning_rate,
            depth=depth,
            loss_function="MultiClass",
            eval_metric="Accuracy",
            random_seed=random_seed,
            verbose=100
        )

        # -----------------------------
        # Train model
        # -----------------------------
        model.fit(
            self.X_train,
            self.y_train,
            eval_set=(
                self.X_test,
                self.y_test
            ),
            use_best_model=True,
            early_stopping_rounds=early_stopping_rounds
        )

        # -----------------------------
        # Store model
        # -----------------------------
        self.model = model
        self.class_labels = model.classes_
        self.training_history = model.get_evals_result()

        # -----------------------------
        # Training summary
        # -----------------------------
        print("=" * 50)
        print("Model training completed.")
        print("=" * 50)

        print(f"Best Iteration : {model.get_best_iteration()}")
        print(f"Best Score     : {model.get_best_score()}")

        print("\nClass Labels:")
        print(self.class_labels)


        return self.model

    # Evaluate the model metrics
    def evaluate_model(self):
        """
        Evaluate the trained model.

        Stores:
            self.y_pred
            self.y_pred_proba
            self.evaluation_results
        """

        # --------------------------------
        # Validation
        # --------------------------------

        if self.model is None:
            raise ValueError(
                "Model not trained. Run train_model() first."
            )

        # --------------------------------
        # Predictions
        # --------------------------------

        y_pred = self.model.predict(self.X_test)

        # CatBoost may return shape (n,1)
        y_pred = y_pred.flatten()

        y_pred_proba = self.model.predict_proba(self.X_test)

        self.y_pred = y_pred
        self.y_pred_proba = y_pred_proba

        # --------------------------------
        # Metrics
        # --------------------------------

        accuracy = accuracy_score(
            self.y_test,
            y_pred
        )

        balanced_accuracy = balanced_accuracy_score(
            self.y_test,
            y_pred
        )

        precision = precision_score(
            self.y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            self.y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        f1 = f1_score(
            self.y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        cm = confusion_matrix(
            self.y_test,
            y_pred,
            labels=self.class_labels
        )

        report = classification_report(
            self.y_test,
            y_pred,
            labels=self.class_labels,
            zero_division=0
        )

        # --------------------------------
        # Store results
        # --------------------------------

        self.evaluation_results = {

            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1,
            "confusion_matrix": cm,
            "classification_report": report

        }

        # --------------------------------
        # Print Summary
        # --------------------------------

        print("=" * 50)
        print("MODEL EVALUATION")
        print("=" * 50)

        print(f"Accuracy:           {accuracy:.4f}")
        print(f"Balanced Accuracy:  {balanced_accuracy:.4f}")
        print(f"Precision (Macro):  {precision:.4f}")
        print(f"Recall (Macro):     {recall:.4f}")
        print(f"F1 Score (Macro):   {f1:.4f}")

        print("\nConfusion Matrix")
        print(cm)

        print("\nClassification Report")
        print(report)

        return self.evaluation_results

    # Feature importance
    def feature_importance(self):
        """
        Calculate and store feature importance.

        Stores:
            self.feature_importance_df
        """

        # --------------------------------
        # Validation
        # --------------------------------

        if self.model is None:
            raise ValueError(
                "Model not trained. Run train_model() first."
            )

        # --------------------------------
        # Calculate importance
        # --------------------------------

        importance = self.model.get_feature_importance()

        importance_df = pd.DataFrame({

            "feature": self.feature_columns,
            "importance": importance

        })

        importance_df = (
            importance_df
            .sort_values(
                by="importance",
                ascending=False
            )
            .reset_index(drop=True)
        )

        # --------------------------------
        # Store
        # --------------------------------

        self.feature_importance_df = importance_df
        self.top_features = importance_df.head(10)

        # --------------------------------
        # Display
        # --------------------------------

        print("=" * 50)
        print("FEATURE IMPORTANCE")
        print("=" * 50)

        print(importance_df)

        return importance_df

    # save the model to be accessed via an api
    def save_model(self, save_dir="worldcup_predictor"):
        """
        Save the trained model and supporting artifacts.

        Saves:
            - CatBoost model
            - Feature columns
            - Class labels
            - Current Elo ratings
            - Team history
            - Head-to-head history
            - Metadata
        """

        # --------------------------------
        # Validation
        # --------------------------------
        if self.model is None:
            raise ValueError(
                "No trained model found. Run train_model() first."
            )

        # --------------------------------
        # Create directory
        # --------------------------------
        os.makedirs(save_dir, exist_ok=True)

        # --------------------------------
        # Save CatBoost model
        # --------------------------------
        self.model.save_model(
            os.path.join(
                save_dir,
                "catboost_model.cbm"
            )
        )

        # --------------------------------
        # Save supporting objects
        # --------------------------------
        with open(
            os.path.join(
                save_dir,
                "feature_columns.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(self.feature_columns, f)

        with open(
            os.path.join(
                save_dir,
                "class_labels.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(self.class_labels, f)

        with open(
            os.path.join(
                save_dir,
                "current_elo.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(self.current_elo, f)

        with open(
            os.path.join(
                save_dir,
                "team_history.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(self.team_history, f)

        with open(
            os.path.join(
                save_dir,
                "h2h_history.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(self.h2h_history, f)

        # --------------------------------
        # Metadata
        # --------------------------------
        metadata = {

            "train_years": [
                int(year)
                for year in self.train_years
            ],

            "test_years": [
                int(year)
                for year in self.test_years
            ],

            "n_features": int(
                len(self.feature_columns)
            ),

            "feature_names": list(
                self.feature_columns
            ),

            "model_type": "CatBoostClassifier",

            "target_mapping": {

                "-1": "Team B Win",

                "0": "Draw",

                "1": "Team A Win"

            }

        }

        with open(
            os.path.join(
                save_dir,
                "metadata.json"
            ),
            "w"
        ) as f:
            json.dump(metadata, f, indent=4)

        print("=" * 50)
        print("Model successfully saved.")
        print("=" * 50)
        print(f"Location: {save_dir}")

        return save_dir

    # Get a snapshot of the team to predict
    def get_team_snapshot(self, team_name):
        """
        Build the latest snapshot for a team.
        """

        # --------------------------
        # Elo
        # --------------------------

        if hasattr(self, "team_snapshots"):

            if team_name in self.team_snapshots:
                return self.team_snapshots[team_name]

        elo = self.current_elo.get(
            team_name,
            1500
        )

        # --------------------------
        # Rolling Form
        # --------------------------

        matches = self.team_history.get(
            team_name,
            []
        )

        recent = matches[-5:]

        if len(recent) == 0:

            win_rate = 0.5
            goals = 1.0
            conceded = 1.0
            goal_diff = 0.0

        else:

            win_rate = sum(
                x["result"]
                for x in recent
            ) / len(recent)

            goals = sum(
                x["gf"]
                for x in recent
            ) / len(recent)

            conceded = sum(
                x["ga"]
                for x in recent
            ) / len(recent)

            goal_diff = goals - conceded

        # --------------------------
        # Historical Tournament Data
        # --------------------------

        team_df = self.df_teams[
            self.df_teams["name"] == team_name
        ]

        if len(team_df) == 0:

            titles = 0
            knockout_rate = 0
            average_finish = 32
            appearances = 0
            semi_final_rate = 0
            final_rate = 0

        else:

            appearances = len(team_df)

            titles = int(
                (team_df["finalPosition"] == 1).sum()
            )

            knockout_rate = float(
                (team_df["finalPosition"] <= 16).mean()
            )

            average_finish = float(
                team_df["finalPosition"].mean()
            )

            semi_final_rate = float(
                (team_df["finalPosition"] <= 4).mean()
            )

            final_rate = float(
                (team_df["finalPosition"] <= 2).mean()
            )

        # --------------------------
        # Final Snapshot
        # --------------------------

        snapshot = {

            "elo": elo,

            "win_rate_last_5": win_rate,

            "goals_last_5": goals,

            "conceded_last_5": conceded,

            "goal_diff_last_5": goal_diff,

            "historical_titles": titles,

            "knockout_rate": knockout_rate,

            "average_finish": average_finish,

            "world_cup_appearances": appearances,

            "semi_final_rate": semi_final_rate,

            "final_rate": final_rate

        }
        self.team_snapshots[team_name] = snapshot

        return snapshot

    # get match features for predicting from a new sample
    def build_match_features(
        self,
        team_a,
        team_b,
        stage_weight=1
    ):
        """
        Build model features for a future match.

        Parameters
        ----------
        team_a : str

        team_b : str

        stage_weight : int
            Group = 1
            R16   = 2
            QF    = 3
            SF    = 4
            Third = 4
            Final = 5

        Returns
        -------
        pd.DataFrame
        """

        # --------------------------------
        # Team snapshots
        # --------------------------------

        home = self.get_team_snapshot(
            team_a
        )

        away = self.get_team_snapshot(
            team_b
        )

        # --------------------------------
        # Head-to-head
        # --------------------------------

        pair = tuple(
            sorted(
                [team_a, team_b]
            )
        )

        h2h_matches = self.h2h_history.get(
            pair,
            []
        )

        if len(h2h_matches) == 0:

            h2h_games = 0
            h2h_win_rate = 0.5
            h2h_goal_diff = 0.0

        else:

            h2h_games = len(
                h2h_matches
            )

            h2h_win_rate = sum(

                x["result"]

                for x in h2h_matches

            ) / h2h_games

            h2h_goal_diff = sum(x["goal_diff"] for x in h2h_matches) / h2h_games

        # --------------------------------
        # Build feature row
        # --------------------------------

        row = {

            "home_team_elo":
                home["elo"],

            "away_team_elo":
                away["elo"],

            "elo_difference":
                home["elo"] - away["elo"],

            "home_win_rate_last_5":
                home["win_rate_last_5"],

            "away_win_rate_last_5":
                away["win_rate_last_5"],

            "home_goals_last_5":
                home["goals_last_5"],

            "away_goals_last_5":
                away["goals_last_5"],

            "home_conceded_last_5":
                home["conceded_last_5"],

            "away_conceded_last_5":
                away["conceded_last_5"],

            "home_goal_diff_last_5":
                home["goal_diff_last_5"],

            "away_goal_diff_last_5":
                away["goal_diff_last_5"],

            "head_to_head_games":
                h2h_games,

            "head_to_head_win_rate":
                h2h_win_rate,

            "head_to_head_goal_difference":
                h2h_goal_diff,

            "home_historical_titles":
                home["historical_titles"],

            "away_historical_titles":
                away["historical_titles"],

            "home_knockout_rate":
                home["knockout_rate"],

            "away_knockout_rate":
                away["knockout_rate"],

            "home_average_finish":
                home["average_finish"],

            "away_average_finish":
                away["average_finish"],

            "home_world_cup_appearances":
                home["world_cup_appearances"],

            "away_world_cup_appearances":
                away["world_cup_appearances"],

            "home_semi_final_rate":
                home["semi_final_rate"],

            "away_semi_final_rate":
                away["semi_final_rate"],

            "home_final_rate":
                home["final_rate"],

            "away_final_rate":
                away["final_rate"],

            "stage_weight":
                stage_weight

        }

        # --------------------------------
        # Create DataFrame
        # --------------------------------

        features = pd.DataFrame(
            [row]
        )

        # --------------------------------
        # Ensure exact order
        # --------------------------------

        features = features[
            self.feature_columns
        ]
        missing = set(
            self.feature_columns
        ) - set(
            features.columns
        )

        if missing:

            raise ValueError(
                f"Missing features: {missing}"
            )

        features = features[
            self.feature_columns
        ]

        return features

    # Predict match
    def predict_match(
        self,
        team_a,
        team_b,
        stage_weight=1
    ):
        """
        Predict the outcome of a match.

        Parameters
        ----------
        team_a : str

        team_b : str

        stage_weight : int

        Returns
        -------
        dict
        """

        # -----------------------------
        # Validation
        # -----------------------------

        if self.model is None:
            raise ValueError(
                "Model not trained. Run train_model() first."
            )

        # -----------------------------
        # Build features
        # -----------------------------

        features = self.build_match_features(

            team_a,

            team_b,

            stage_weight

        )

        # -----------------------------
        # Predict
        # -----------------------------

        prediction = self.model.predict(
            features
        )

        # CatBoost returns [[1]]
        prediction = int(
            prediction.flatten()[0]
        )

        # -----------------------------
        # Human-readable label
        # -----------------------------

        if prediction == 1:

            result = f"{team_a} Win"

        elif prediction == -1:

            result = f"{team_b} Win"

        else:

            result = "Draw"

        # -----------------------------
        # Output
        # -----------------------------

        output = {

            "team_a": team_a,

            "team_b": team_b,

            "prediction": prediction,

            "result": result

        }
        self.last_prediction = output
        return output

    # predict match probabilities
    def predict_match_proba(
        self,
        team_a,
        team_b,
        stage_weight=1
    ):
        """
        Predict match probabilities.

        Returns
        -------
        dict
        """

        # ----------------------------
        # Validation
        # ----------------------------

        if self.model is None:
            raise ValueError(
                "Model not trained."
            )

        # ----------------------------
        # Build features
        # ----------------------------

        features = self.build_match_features(

            team_a,

            team_b,

            stage_weight

        )

        # ----------------------------
        # Predict probabilities
        # ----------------------------

        probabilities = self.model.predict_proba(
            features
        )[0]

        # ----------------------------
        # Build result
        # ----------------------------

        output = {

            "team_a": team_a,

            "team_b": team_b,

            "probabilities": {}

        }

        for label, probability in zip(

            self.class_labels,

            probabilities

        ):

            output["probabilities"][
                int(label)
            ] = float(
                probability
            )

        # ----------------------------
        # Human-readable labels
        # ----------------------------

        output["team_a_win"] = output[
            "probabilities"
        ].get(
            1,
            0.0
        )

        output["draw"] = output[
            "probabilities"
        ].get(
            0,
            0.0
        )

        output["team_b_win"] = output[
            "probabilities"
        ].get(
            -1,
            0.0
        )

        output["classes"] = list(
            map(int, self.class_labels)
        )

        output["values"] = list(
            map(float, probabilities)
        )

        output["classes"] = [
            int(x)
            for x in self.class_labels
        ]

        output["values"] = [
            float(x)
            for x in probabilities
        ]

        return output

    #load the world cup 2026 teams
    def load_2026_teams(self):
        """
        Load the predefined FIFA 2026 groups.
        """

        self.groups = {
            "A": ["Czechia", "Mexico", "South Korea", "South Africa"],
            "B": ["Bosnia and Herzegovina", "Canada", "Qatar", "Switzerland"],
            "C": ["Brazil", "Haiti", "Morocco", "Scotland"],
            "D": ["Australia", "Paraguay", "Turkiye", "USA"],
            "E": ["Curacao", "Ecuador", "Germany", "Ivory Coast"],
            "F": ["Japan", "Netherlands", "Sweden", "Tunisia"],
            "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
            "H": ["Cape Verde", "Saudi Arabia", "Spain", "Uruguay"],
            "I": ["France", "Iraq", "Norway", "Senegal"],
            "J": ["Algeria", "Argentina", "Austria", "Jordan"],
            "K": ["Colombia", "DR Congo", "Portugal", "Uzbekistan"],
            "L": ["Croatia", "England", "Ghana", "Panama"]
        }

        # --------------------------
        # Initialize Group Tables
        # --------------------------
        self.group_tables = {}

        for group_name, teams in self.groups.items():
            self.group_tables[group_name] = {}
            for team in teams:
                self.group_tables[group_name][team] = {
                    "played": 0,
                    "wins": 0,
                    "draws": 0,
                    "losses": 0,
                    "goals_for": 0,
                    "goals_against": 0,
                    "goal_difference": 0,
                    "points": 0
                }

        self.knockout_bracket = {}
        self.stage_weights = {
            "group": 1,
            "round_of_32": 2,
            "round_of_16": 3,
            "quarter_final": 4,
            "semi_final": 5,
            "third_place": 5,
            "final": 6
        }

        print("=" * 50)
        print("WORLD CUP 2026 LOADED")
        print("=" * 50)
        print(f"Groups: {len(self.groups)}")
        print(f"Teams : {sum(len(x) for x in self.groups.values())}")

        return self.groups

    # simulate group match
    def simulate_group_match(self,team_a,team_b):
        """
        Simulate one group stage match.
        """

        # --------------------------
        # Predict probabilities
        # --------------------------
        proba = self.predict_match_proba(
            team_a,
            team_b,
            stage_weight=self.stage_weights["group"]
        )

        # --------------------------
        # Sample outcome
        # --------------------------
        outcome = np.random.choice(
            proba["classes"],
            p=proba["values"]
        )

        # --------------------------
        # Generate score
        # --------------------------
        if outcome == 1:
            scores = [
                (1, 0),
                (2, 0),
                (2, 1),
                (3, 1),
                (3, 0)
            ]
        elif outcome == 0:
            scores = [
                (0, 0),
                (1, 1),
                (2, 2)
            ]
        else:
            scores = [
                (0, 1),
                (0, 2),
                (1, 2),
                (1, 3),
                (0, 3)
            ]

        home_goals, away_goals = scores[
            np.random.randint(len(scores))
        ]

        # --------------------------
        # Update standings
        # --------------------------
        self._update_group_table(
            team_a,
            team_b,
            home_goals,
            away_goals
        )
        # --------------------------
        # Return
        # --------------------------
        return {
            "team_a": team_a,
            "team_b": team_b,
            "home_goals": home_goals,
            "away_goals": away_goals,
            "outcome": int(outcome)
        }
    #a private helper function to update group table
    def _update_group_table(
        self,
        team_a,
        team_b,
        goals_a,
        goals_b
    ):
        group = None
        for g, teams in self.groups.items():
            if (
                team_a in teams and
                team_b in teams
            ):
                group = g
                break

        if group is None:
            raise ValueError(
                "Teams not in same group."
            )

        table = self.group_tables[group]

        a = table[team_a]
        b = table[team_b]

        a["played"] += 1
        b["played"] += 1

        a["goals_for"] += goals_a
        a["goals_against"] += goals_b

        b["goals_for"] += goals_b
        b["goals_against"] += goals_a

        a["goal_difference"] = (
            a["goals_for"]
            - a["goals_against"]
        )
        b["goal_difference"] = (
            b["goals_for"]
            - b["goals_against"]
        )

        if goals_a > goals_b:
            a["wins"] += 1
            b["losses"] += 1
            a["points"] += 3
        elif goals_a < goals_b:
            b["wins"] += 1
            a["losses"] += 1
            b["points"] += 3
        else:
            a["draws"] += 1
            b["draws"] += 1
            a["points"] += 1
            b["points"] += 1

    # Simulate Group Matches
    def simulate_group(self, group_name):
        """
        Simulate an entire group.

        Parameters
        ----------
        group_name : str

        Returns
        -------
        list
            Simulated matches.
        """

        # ------------------------
        # Validation
        # ------------------------
        if self.groups is None:
            raise ValueError(
                "Run load_2026_teams() first."
            )

        if group_name not in self.groups:
            raise ValueError(
                f"Group {group_name} not found."
            )

        teams = self.groups[group_name]

        # ------------------------
        # Store group matches
        # ------------------------

        if not hasattr(self, "group_matches"):
            self.group_matches = {}

        self.group_matches[group_name] = []

        # ------------------------
        # Fixtures
        # ------------------------
        fixtures = list(combinations(teams,2))

        # ------------------------
        # Simulate Matches
        # ------------------------
        if not hasattr(self, "group_matches"):
            self.group_matches = {}

        self.group_matches[group_name] = []

        results = []

        for team_a, team_b in fixtures:

            match = self.simulate_group_match(
                team_a,
                team_b
            )

            results.append(match)

            self.group_matches[group_name].append(
                match
            )

            self.group_matches[group_name].append(
                {
                    "home": team_a,
                    "away": team_b,
                    "home_goals": match["home_goals"],
                    "away_goals": match["away_goals"]
                }
            )
        self.group_matches[group_name].append(match)

        return {
                "group": group_name,
                "matches": results,
                "table": self.build_group_table(group_name)
            }

    # group match tables
    def build_group_table(self, group_name):
        """
        Build and sort a group table.

        Parameters
        ----------
        group_name : str

        Returns
        -------
        list
        """

        if self.group_tables is None:
            raise ValueError(
                "Run load_2026_teams() first."
            )

        if group_name not in self.group_tables:
            raise ValueError(
                f"Group {group_name} not found."
            )

        table = self.group_tables[group_name]
        standings = []

        for team, stats in table.items():
            row = {
                "team": team,
                "played": stats["played"],
                "wins": stats["wins"],
                "draws": stats["draws"],
                "losses": stats["losses"],
                "goals_for": stats["goals_for"],
                "goals_against": stats["goals_against"],
                "goal_difference": stats["goal_difference"],
                "points": stats["points"]
            }
            standings.append(row)

        # --------------------------------
        # Sort by FIFA rules
        # --------------------------------
        standings = sorted(
            standings,
            key=lambda x: (
                x["points"],
                x["goal_difference"],
                x["goals_for"]
            ),
            reverse=True
        )

        # --------------------------------
        # Add positions
        # --------------------------------
        for i, row in enumerate(standings):
            row["position"] = i + 1

        return pd.DataFrame(standings)

    #another helper function
    def get_group_qualifiers(self, group_name):
        """
        Extract the top two qualifying teams from a group table.
        """
        table = self.build_group_table(group_name)

        # Using standard list indexing since build_group_table returns a list
        return {
            "winner": table[0]["team"],
            "runner_up": table[1]["team"]
        }
    #Create a private helper function for the group matches head to head
    def _calculate_head_to_head(self, tied_teams, matches):
        stats = {}

        for team in tied_teams:
            stats[team] = {"points": 0, "gd": 0, "gf": 0}

        for match in matches:
            home = match["home"]
            away = match["away"]

            if home not in tied_teams or away not in tied_teams:
                continue

            hg = match["home_goals"]
            ag = match["away_goals"]

            stats[home]["gf"] += hg
            stats[home]["gd"] += hg - ag

            stats[away]["gf"] += ag
            stats[away]["gd"] += ag - hg

            if hg > ag:
                stats[home]["points"] += 3
            elif ag > hg:
                stats[away]["points"] += 3
            else:
                stats[home]["points"] += 1
                stats[away]["points"] += 1

        return stats

    #bulding fifa 2026 standings
    def build_group_table(self, group_name):
        """Build official FIFA World Cup 2026 standings."""
        table = self.group_tables[group_name]

        standings = list(table.values())

        # First sort using overall statistics
        standings.sort(
                key=lambda x: (
                    x["points"],
                    x["goal_difference"],
                    x["goals_for"]
                ),
                reverse=True
            )

        final_table = []
        i = 0

        while i < len(standings):
            tied = [standings[i]]
            j = i + 1

            while (
                j < len(standings)
                and standings[j]["points"] == standings[i]["points"]
                and standings[j]["gd"] == standings[i]["gd"]
                and standings[j]["gf"] == standings[i]["gf"]
            ):
                tied.append(standings[j])
                j += 1

            # Resolve tie
            if len(tied) > 1:
                tied_team_names = [team["team"] for team in tied]

                h2h = self._calculate_head_to_head(
                    tied_team_names, self.group_matches[group_name]
                )

                tied.sort(
                    key=lambda x: (
                        h2h[x["team"]]["points"],
                        h2h[x["team"]]["gd"],
                        h2h[x["team"]]["gf"],
                        x["gd"],
                        x["gf"],
                        -x.get("fair_play", 0),
                        -x.get("fifa_rank", 999),
                    ),
                    reverse=True,
                )

            final_table.extend(tied)
            i = j

        return final_table

In [39]:
predictor = WorldCupPredictor()

predictor.load_data(
    df_matches,
    df_teams,
    df_tournaments
)

Data loaded successfully
Matches: (1068, 28)
Teams: (537, 16)
Tournaments: (23, 4)


In [40]:
predictor.load_2026_teams()

WORLD CUP 2026 LOADED
Groups: 12
Teams : 48


{'A': ['Czechia', 'Mexico', 'South Korea', 'South Africa'],
 'B': ['Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland'],
 'C': ['Brazil', 'Haiti', 'Morocco', 'Scotland'],
 'D': ['Australia', 'Paraguay', 'Turkiye', 'USA'],
 'E': ['Curacao', 'Ecuador', 'Germany', 'Ivory Coast'],
 'F': ['Japan', 'Netherlands', 'Sweden', 'Tunisia'],
 'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
 'H': ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay'],
 'I': ['France', 'Iraq', 'Norway', 'Senegal'],
 'J': ['Algeria', 'Argentina', 'Austria', 'Jordan'],
 'K': ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan'],
 'L': ['Croatia', 'England', 'Ghana', 'Panama']}

In [41]:
print(predictor.groups)

{'A': ['Czechia', 'Mexico', 'South Korea', 'South Africa'], 'B': ['Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland'], 'C': ['Brazil', 'Haiti', 'Morocco', 'Scotland'], 'D': ['Australia', 'Paraguay', 'Turkiye', 'USA'], 'E': ['Curacao', 'Ecuador', 'Germany', 'Ivory Coast'], 'F': ['Japan', 'Netherlands', 'Sweden', 'Tunisia'], 'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'], 'H': ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay'], 'I': ['France', 'Iraq', 'Norway', 'Senegal'], 'J': ['Algeria', 'Argentina', 'Austria', 'Jordan'], 'K': ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan'], 'L': ['Croatia', 'England', 'Ghana', 'Panama']}


In [42]:
for group, teams in predictor.groups.items():
    print(group, len(teams))

A 4
B 4
C 4
D 4
E 4
F 4
G 4
H 4
I 4
J 4
K 4
L 4


In [43]:
result = predictor.simulate_group("C")

print(result)

ValueError: Model not trained.

In [44]:
for match in result["matches"]:
    print(match)

NameError: name 'result' is not defined

In [30]:
predictor.clean_data()

predictor.remove_future_matches()

predictor.standardize_team_names()

predictor.validate_data()

predictor.build_target()

predictor.sort_matches()

predictor.calculate_elo()

predictor.build_rolling_features()

predictor.build_head_to_head()

predictor.build_historical_features()

predictor.build_stage_weight()

predictor.build_training_dataset()

predictor.select_features()

predictor.split_data()

predictor.train_model()

predictor.evaluate_model()

predictor.feature_importance()

predictor.save_model()

predictor.get_team_snapshot("Brazil")


Cleaning completed
Removed 104 future matches
Remaining matches: 964
Team names standardized
MATCHES
(964, 28)
id                   0
date                 0
kickoff              0
stage                0
homeTeam             0
awayTeam             0
homeScore            0
awayScore            0
result               0
extraTime            0
penalties          964
stadium              0
stadiumId            0
city                 0
attendance         964
referee              0
weather             53
stageNormalized    238
tournament_year      0
captains           961
goals              961
penaltyShootout    961
matchNo            964
kickoffUtc         964
timezone           964
homeRef            964
awayRef            964
country            964
dtype: int64
TEAMS
(537, 16)
name                     0
code                     0
iso                      0
confederation            0
groupStage              31
knockoutPath             0
finalPosition           48
coach                    0


{'elo': 1695.4467501354318,
 'win_rate_last_5': 0.7,
 'goals_last_5': 1.6,
 'conceded_last_5': 0.6,
 'goal_diff_last_5': 1.0,
 'historical_titles': 5,
 'knockout_rate': 0.9565217391304348,
 'average_finish': 4.590909090909091,
 'world_cup_appearances': 23,
 'semi_final_rate': 0.4782608695652174,
 'final_rate': 0.30434782608695654}

In [89]:
features = predictor.build_match_features(

    "Brazil",

    "France",

    stage_weight=5

)

print(features.shape)

print(features.columns.tolist())

print(features)

(1, 27)
['home_team_elo', 'away_team_elo', 'elo_difference', 'home_win_rate_last_5', 'away_win_rate_last_5', 'home_goals_last_5', 'away_goals_last_5', 'home_conceded_last_5', 'away_conceded_last_5', 'home_goal_diff_last_5', 'away_goal_diff_last_5', 'head_to_head_games', 'head_to_head_win_rate', 'head_to_head_goal_difference', 'home_historical_titles', 'away_historical_titles', 'home_knockout_rate', 'away_knockout_rate', 'home_average_finish', 'away_average_finish', 'home_world_cup_appearances', 'away_world_cup_appearances', 'home_semi_final_rate', 'away_semi_final_rate', 'home_final_rate', 'away_final_rate', 'stage_weight']
   home_team_elo  away_team_elo  elo_difference  home_win_rate_last_5  \
0     1695.44675    1694.472056        0.974694                   0.7   

   away_win_rate_last_5  home_goals_last_5  away_goals_last_5  \
0                   0.7                1.6                2.0   

   home_conceded_last_5  away_conceded_last_5  home_goal_diff_last_5  ...  \
0            

In [91]:
predictor.load_2026_teams()

match = predictor.simulate_group_match(

    "Brazil",

    "Morocco"

)

print(match)

WORLD CUP 2026 LOADED
Groups: 12
Teams : 48
{'team_a': 'Brazil', 'team_b': 'Morocco', 'home_goals': 2, 'away_goals': 2, 'outcome': 0}


In [92]:
print(
    predictor.group_tables["C"]
)

{'Brazil': {'played': 1, 'wins': 0, 'draws': 1, 'losses': 0, 'goals_for': 2, 'goals_against': 2, 'goal_difference': 0, 'points': 1}, 'Haiti': {'played': 0, 'wins': 0, 'draws': 0, 'losses': 0, 'goals_for': 0, 'goals_against': 0, 'goal_difference': 0, 'points': 0}, 'Morocco': {'played': 1, 'wins': 0, 'draws': 1, 'losses': 0, 'goals_for': 2, 'goals_against': 2, 'goal_difference': 0, 'points': 1}, 'Scotland': {'played': 0, 'wins': 0, 'draws': 0, 'losses': 0, 'goals_for': 0, 'goals_against': 0, 'goal_difference': 0, 'points': 0}}


In [93]:
predictor.load_2026_teams()

matches = predictor.simulate_group("C")

for match in matches:

    print(match)

WORLD CUP 2026 LOADED
Groups: 12
Teams : 48
group
matches
table


In [94]:
predictor.simulate_group("C")

table = predictor.build_group_table("C")

for row in table:

    print(row)

team
played
wins
draws
losses
goals_for
goals_against
goal_difference
points
position


In [95]:
group = predictor.simulate_group("C")

print(group["matches"])

print(group["table"])

[{'team_a': 'Brazil', 'team_b': 'Haiti', 'home_goals': 1, 'away_goals': 3, 'outcome': -1}, {'team_a': 'Brazil', 'team_b': 'Morocco', 'home_goals': 2, 'away_goals': 0, 'outcome': 1}, {'team_a': 'Brazil', 'team_b': 'Scotland', 'home_goals': 2, 'away_goals': 2, 'outcome': 0}, {'team_a': 'Haiti', 'team_b': 'Morocco', 'home_goals': 2, 'away_goals': 2, 'outcome': 0}, {'team_a': 'Haiti', 'team_b': 'Scotland', 'home_goals': 3, 'away_goals': 0, 'outcome': 1}, {'team_a': 'Morocco', 'team_b': 'Scotland', 'home_goals': 1, 'away_goals': 1, 'outcome': 0}]
{'Brazil': {'played': 9, 'wins': 3, 'draws': 2, 'losses': 4, 'goals_for': 11, 'goals_against': 14, 'goal_difference': -3, 'points': 11}, 'Haiti': {'played': 9, 'wins': 4, 'draws': 4, 'losses': 1, 'goals_for': 12, 'goals_against': 7, 'goal_difference': 5, 'points': 16}, 'Morocco': {'played': 9, 'wins': 3, 'draws': 3, 'losses': 3, 'goals_for': 13, 'goals_against': 12, 'goal_difference': 1, 'points': 12}, 'Scotland': {'played': 9, 'wins': 1, 'draws': 

In [96]:
table = predictor.build_group_table("C")

print(table)

       team  played  wins  draws  losses  goals_for  goals_against  \
0     Haiti       9     4      4       1         12              7   
1   Morocco       9     3      3       3         13             12   
2    Brazil       9     3      2       4         11             14   
3  Scotland       9     1      5       3          9             12   

   goal_difference  points  position  
0                5      16         1  
1                1      12         2  
2               -3      11         3  
3               -3       8         4  
